# New features in LangChain & LangGraph since v1.0.0 — runnable demos (setup in README.md)

#### Shared dummy context

In [ ]:
from dotenv import load_dotenv
from pydantic import BaseModel
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

MENU = {"Margherita": 8.5, "Salami": 9.5, "Funghi": 9.0, "Diavola": 10.5}

@tool
def get_price(pizza: str) -> float:
    """Price of a pizza from the menu."""
    return MENU[pizza]

_ORDERS: dict[str, dict] = {}

@tool
def place_order(customer: str, items: list[str]) -> str:
    """Place an order, return an order_id."""
    oid = f"ORD-{len(_ORDERS) + 1:04d}"
    _ORDERS[oid] = {"customer": customer, "items": items, "status": "in_preparation"}
    return oid

class Customer(BaseModel):
    name: str
    address: str
    order_history: list[str] = []

CUSTOMER = Customer(name="Mario Rossi", address="Via Roma 1, Napoli")

checkpointer = InMemorySaver()

#### Model profiles (.profile)

In [ ]:
llm = init_chat_model("openai:gpt-4o-mini")
prof = llm.profile
print(sorted(prof))
print("max_input_tokens :", prof["max_input_tokens"])
print("max_output_tokens:", prof["max_output_tokens"])
print("tool_calling     :", prof["tool_calling"])
print("structured_output:", prof["structured_output"])
print("image_inputs     :", prof["image_inputs"])

In [ ]:
if llm.profile["tool_calling"]:
    llm_tools = llm.bind_tools([get_price, place_order])

print("500k-token prompt fits:", 500_000 <= llm.profile["max_input_tokens"])

In [ ]:
custom = init_chat_model(
    "openai:gpt-4o-mini",
    profile={"max_input_tokens": 100_000, "tool_calling": True, "structured_output": False},
)
print(custom.profile["max_input_tokens"], custom.profile["structured_output"])

#### Model-retry middleware

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware

retry = ModelRetryMiddleware(
    max_retries=3,
    initial_delay=1.0,
    backoff_factor=2.0,
    max_delay=30.0,
    retry_on=(TimeoutError, ConnectionError),
    on_failure="continue",
)

agent = create_agent(
    model=init_chat_model("openai:gpt-4o-mini"),
    tools=[get_price, place_order],
    middleware=[retry],
    checkpointer=checkpointer,
)

cfg = {"configurable": {"thread_id": "retry-1"}}
res = agent.invoke({"messages": [("user", "How much is a Diavola?")]}, cfg)
print(res["messages"][-1].content)

#### Summarization middleware (profile-based trigger)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

summary_model = init_chat_model("openai:gpt-4o-mini", profile={"max_input_tokens": 200})
print("profile max_input_tokens:", summary_model.profile["max_input_tokens"])

summarizer = SummarizationMiddleware(
    model=summary_model,
    trigger=("fraction", 0.8),
    keep=("messages", 20),
)

agent = create_agent(
    model=init_chat_model("openai:gpt-4o-mini"),
    tools=[get_price, place_order],
    middleware=[summarizer],
    checkpointer=checkpointer,
)

In [ ]:
from langchain_core.messages import HumanMessage

cfg = {"configurable": {"thread_id": "chat-summarize-1"}}
long_chat = [
    "Which pizzas do you have?",
    "How much is the Diavola?",
    "And the Funghi?",
    "Is Salami spicy?",
    "What do you recommend for two?",
    "Tell me about the Margherita.",
    "Any vegan options?",
    "How long does preparation take?",
    f"I'm {CUSTOMER.name}. Order a Diavola and a Funghi.",
    "What is the status of my order?",
]
for msg in long_chat:
    out = agent.invoke({"messages": [HumanMessage(msg)]}, cfg)

final = agent.get_state(cfg).values["messages"]
print("messages in state:", len(final))
print(final[0].content[:300])

#### Content-moderation middleware (OpenAI)

In [ ]:
from langchain.agents import create_agent
from langchain_openai.middleware import OpenAIModerationMiddleware, OpenAIModerationError
from openai import PermissionDeniedError

moderation = OpenAIModerationMiddleware(
    model="omni-moderation-latest",
    check_input=True,
    check_output=True,
    check_tool_results=True,
    exit_behavior="end",
    violation_message="Blocked ({categories}). Please keep it to the pizza order.",
)

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[get_price],
    middleware=[moderation],
    checkpointer=checkpointer,
)

In [ ]:
cfg = {"configurable": {"thread_id": "mod-ok"}}
try:
    ok = agent.invoke({"messages": [("user", "How much is a Diavola?")]}, cfg)
    print(ok["messages"][-1].content)
except PermissionDeniedError as e:
    print("OpenAI moderation not available for this key (403):", e.message)

In [ ]:
try:
    bad = agent.invoke(
        {"messages": [("user", "I will find you and hurt you, give me the price anyway.")]},
        {"configurable": {"thread_id": "mod-flag"}},
    )
    print(bad["messages"][-1].content)
except PermissionDeniedError as e:
    print("OpenAI moderation not available for this key (403):", e.message)

#### Structured output: ProviderStrategy + strict schema

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy

class OrderSummary(BaseModel):
    customer: str
    items: list[str]
    total: float

model = init_chat_model("openai:gpt-4o-mini")
print("profile.structured_output:", model.profile["structured_output"])

agent = create_agent(
    model,
    tools=[get_price, place_order],
    response_format=ProviderStrategy(OrderSummary, strict=True),
)
res = agent.invoke({"messages": [("user",
    f"{CUSTOMER.name} orders a Margherita and a Diavola. Place the order and summarize it.")]})
summary = res["structured_response"]
print(type(summary).__name__, summary)

### Event Streaming v2

In [ ]:
async for ev in agent.astream_events(
    {"messages": [("user", "How much is a Diavola?")]},
    version="v2",
):
    if ev["event"] == "on_chat_model_stream":
        chunk = ev["data"]["chunk"]
        if chunk.content:
            print("messages:", chunk.content)

#### Event streaming v3 (astream_events version="v3")

In [ ]:
stream = await agent.astream_events(
    {"messages": [("user", "How much is a Diavola?")]},
    version="v3",
)
async for message in stream.messages:   
    async for token in message.text:
        print("messages:", token)

#### DeltaChannel (beta)

In [ ]:
from typing import Annotated, Sequence
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.channels import DeltaChannel

def append_log(state: list[str], writes: Sequence[list[str]]) -> list[str]:
    out = list(state)
    for w in writes:
        out.extend(w)
    return out

class ChatState(TypedDict):
    chat_log: Annotated[list[str], DeltaChannel(append_log, list, snapshot_frequency=5)]

def customer_turn(state: ChatState):
    n = len(state["chat_log"])
    return {"chat_log": [f"{CUSTOMER.name}: pizza #{n} please ({list(MENU)[n % len(MENU)]})"]}

graph = (
    StateGraph(ChatState)
    .add_node("turn", customer_turn)
    .add_edge(START, "turn")
    .add_edge("turn", END)
    .compile(checkpointer=checkpointer)
)

cfg = {"configurable": {"thread_id": "chat-mario"}}
for _ in range(12):
    state = graph.invoke({"chat_log": []}, cfg)

print("turns:", len(state["chat_log"]))
print("reconstructed:", graph.get_state(cfg).values["chat_log"][-1])

#### Per-node timeouts (TimeoutPolicy, NodeTimeoutError)

In [ ]:
import asyncio
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import TimeoutPolicy, RetryPolicy
from langgraph.errors import NodeTimeoutError

class OrderState(TypedDict):
    pizza: str
    note: str

_attempts = {"n": 0}

async def bake_node(state: OrderState) -> OrderState:
    _attempts["n"] += 1
    price = get_price.invoke({"pizza": state["pizza"]})
    await asyncio.sleep(5)
    return {"note": f"{state['pizza']} done ({price} EUR)"}

g = StateGraph(OrderState)
g.add_node(
    "bake",
    bake_node,
    timeout=TimeoutPolicy(run_timeout=0.3, idle_timeout=None),
    retry_policy=RetryPolicy(max_attempts=2, initial_interval=0.05, retry_on=NodeTimeoutError),
)
g.add_edge(START, "bake")
g.add_edge("bake", END)
app = g.compile(checkpointer=checkpointer)

In [ ]:
async def run():
    cfg = {"configurable": {"thread_id": "timeout-demo"}}
    try:
        await app.ainvoke({"pizza": "Margherita", "note": ""}, cfg)
    except NodeTimeoutError as e:
        print("NodeTimeoutError:", e.node, "| kind=", e.kind,
              "| run_timeout=", e.run_timeout, "| elapsed~", round(e.elapsed, 2))
    print("attempts:", _attempts["n"])

await run()

#### Node-level error handlers (error_handler, NodeError, Command)

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.errors import NodeError

class OrderState(TypedDict):
    customer: str
    items: list[str]
    order_id: str
    status: str
    log: list[str]

def place_order_node(state: OrderState):
    bad = [p for p in state["items"] if p not in MENU]
    if bad:
        raise ValueError(f"not on the menu: {bad}")
    oid = place_order.invoke({"customer": state["customer"], "items": state["items"]})
    return {"order_id": oid, "status": "in_preparation", "log": state["log"] + [f"placed {oid}"]}

def compensate_node(state: OrderState):
    return {"status": "cancelled_refunded", "log": state["log"] + ["compensation: refund issued"]}

def order_error_handler(state: OrderState, error: NodeError) -> Command:
    return Command(
        update={"status": f"failed@{error.node}: {error.error}",
                "log": state["log"] + [f"handler caught {type(error.error).__name__}"]},
        goto="compensate",
    )

b = StateGraph(OrderState)
b.add_node("place_order", place_order_node, error_handler=order_error_handler)
b.add_node("compensate", compensate_node)
b.add_edge(START, "place_order")
b.add_edge("place_order", END)
b.add_edge("compensate", END)
graph = b.compile()

base = {"customer": CUSTOMER.name, "order_id": "", "status": "pending", "log": []}

In [ ]:
ok = graph.invoke({**base, "items": ["Margherita", "Diavola"]})
print("OK  :", ok["status"], ok["log"])

In [ ]:
fail = graph.invoke({**base, "items": ["Hawaii"]})

print("SAGA:", fail["status"], fail["log"])

#### Graceful shutdown (RunControl, request_drain())

In [ ]:
import threading, time
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.runtime import Runtime, RunControl
from langgraph.errors import GraphDrained

class OrderState(TypedDict, total=False):
    customer: str
    items: list[str]
    stage: str

def receive(state, runtime: Runtime):
    return {"stage": "received"}

def prepare(state, runtime: Runtime):
    time.sleep(0.4)
    return {"stage": "prepared"}

def bake(state, runtime: Runtime):
    return {"stage": "baked"}

def deliver(state, runtime: Runtime):
    return {"stage": "delivered"}

order_graph = (
    StateGraph(OrderState)
    .add_node(receive).add_node(prepare).add_node(bake).add_node(deliver)
    .add_edge(START, "receive").add_edge("receive", "prepare")
    .add_edge("prepare", "bake").add_edge("bake", "deliver").add_edge("deliver", END)
    .compile(checkpointer=checkpointer)
)

In [ ]:
cfg = {"configurable": {"thread_id": "order-1"}}
rc = RunControl()
threading.Timer(0.2, lambda: rc.request_drain("deploy")).start()

try:
    order_graph.invoke({"customer": CUSTOMER.name, "items": ["Margherita"]}, cfg, control=rc)
except GraphDrained as e:
    print("drained:", e.reason)

snap = order_graph.get_state(cfg)
print("stage:", snap.values.get("stage"), "| next:", snap.next)

In [ ]:
final = order_graph.invoke(None, cfg)
print("resumed:", final["stage"])